# Exploratory Colab experiment

> Cleaned archive of the original graduation-project notebook. For new leakage-aware runs, use the reusable pipeline under src/ and scripts/.


In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, BatchNormalization, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight



In [ ]:
# =========================================================
# 0) Google Drive’ı bağla & GPU bilgisini göster
# =========================================================

import tensorflow as tf, os, sys, numpy as np
print("TF versiyonu:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


In [ ]:
# =========================================================
# 1) Yol tanımları ve genel hiper-parametreler
# =========================================================
BASE_DIR      = 'data/split'   # <== klasör sende farklıysa değiştir
TRAIN_DIR     = os.path.join(BASE_DIR, 'train')
TEST_DIR      = os.path.join(BASE_DIR, 'test')

IMG_SIZE      = 224
BATCH_SIZE    = 32
SEED          = 42

STAGE1_EPOCHS = 10         # yalnız kafa katmanları
STAGE2_EPOCHS = 90          # tüm model ince ayar
LR_STAGE1     = 1e-4
LR_STAGE2     = 1e-5


In [ ]:
# =========================================================
# 2) tf.data + yerleşik data-augmentation pipeline’ı
#    (Keras 3 ile uyumlu, uyarı vermiyor)
# =========================================================
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.15,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Sınıflar:", class_names)

# --- Veri artırma ---
DATA_AUG = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

# --- Performans iyileştirmeleri ---
def cfg(ds, shuffle=True):
    if shuffle: ds = ds.shuffle(1024, seed=SEED)
    return ds.prefetch(AUTOTUNE).cache()

train_ds = cfg(train_ds)
val_ds   = cfg(val_ds, shuffle=False)
test_ds  = cfg(test_ds, shuffle=False)


In [ ]:
# =========================================================
# 3) Sınıf ağırlıkları (dengesiz veri varsa)
# =========================================================
from sklearn.utils import class_weight

y_train_all = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in train_ds])
cw = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_all),
        y=y_train_all
)
class_weights = {i:w for i, w in enumerate(cw)}
print("Class Weights:", class_weights)


In [ ]:
# =========================================================
# 4) Model mimarisi (VGG19 tabanı + GAP + Dense katmanları)
# =========================================================
base_model = tf.keras.applications.VGG19(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False   # aşama-1’de donuk

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = DATA_AUG(inputs)                  # ↑ veri artırma
x       = tf.keras.applications.vgg19.preprocess_input(x)
x       = base_model(x, training=False)     # BN katmanları inference modda
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.BatchNormalization()(x)
x       = tf.keras.layers.Dense(256, activation='relu',
                               kernel_regularizer=tf.keras.regularizers.l2(1e-3))(x)
x       = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()


In [ ]:
# =========================================================
# 5-A) Aşama 1 – Sadece kafa katmanlarını eğit
# =========================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_STAGE1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

ckpt1 = tf.keras.callbacks.ModelCheckpoint(
    'artifacts/best_vgg19_stage1.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
plateau1 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    class_weight=class_weights,
    callbacks=[ckpt1, plateau1]
)


In [ ]:
# =========================================================
# 5-B) Aşama 2 – TÜM katmanları aç, ince ayar yap
#      (son 2 blok hariç açılsın dersen: for l in base_model.layers[-8:]…)
# =========================================================
base_model.trainable = True
# İstersen ağırlıkların ilk %70’ini donuk bırak:
for layer in base_model.layers[:int(0.7*len(base_model.layers))]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_STAGE2),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

ckpt2 = tf.keras.callbacks.ModelCheckpoint(
    'artifacts/best_vgg19_stage2.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
plateau2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    verbose=1,
    min_lr=1e-8
)
# (İster-sen durdurmayı aç)
# early = tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=STAGE1_EPOCHS,
    epochs=STAGE1_EPOCHS+STAGE2_EPOCHS,
    class_weight=class_weights,
    callbacks=[ckpt2, plateau2]   # , early
)


In [ ]:
# =========================================================
# 6) En iyi ağırlıkları yükle + Test kümesi değerlendirmesi
# =========================================================
best_model = tf.keras.models.load_model('artifacts/best_vgg19_stage2.keras')
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}")


In [ ]:
# =========================================================
# 7) Confusion matrix, sınıf raporu (opsiyonel görselleştirme)
# =========================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
y_pred_probs = best_model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_true, y_pred)
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Tahmin"); plt.ylabel("Gerçek"); plt.title("Confusion Matrix")
plt.show()


In [ ]:
# =========================================================
# Test kümesi için accuracy, F1, sensitivity, specificity
# =========================================================
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, recall_score, precision_score

# 1) Modeli yükle
best_model = tf.keras.models.load_model(
    'artifacts/best_vgg19_stage2.keras'
)

# 2) Tahminler
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
y_pred_probs = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# 3) Confusion matrix
cm = confusion_matrix(y_true, y_pred)
# cm[i, j] = gerçek i, tahmin j

# 4) Duyarlılık (sensitivity) & Özgüllük (specificity) hesap fonksiyonu
sens_list, spec_list = [], []
for i in range(len(class_names)):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - (TP + FN + FP)
    sensitivity = TP / (TP + FN + 1e-9)
    specificity = TN / (TN + FP + 1e-9)
    sens_list.append(sensitivity)
    spec_list.append(specificity)

# 5) F1 ve Accuracy
f1_list   = f1_score(y_true, y_pred, average=None)
accuracy  = accuracy_score(y_true, y_pred)
macro_f1  = f1_score(y_true, y_pred, average='macro')
macro_sen = np.mean(sens_list)
macro_spec= np.mean(spec_list)

# 6) Sonuçları tabloya dök
df = pd.DataFrame({
    "Sınıf":       class_names,
    "Accuracy":    [accuracy]*len(class_names),   # global acc tekrar
    "F1":          np.round(f1_list,   3),
    "Sensitivity": np.round(sens_list, 3),
    "Specificity": np.round(spec_list, 3)
})
print(df.to_string(index=False))

print("\n--- Makro Ortalama ---")
print(f"Accuracy  : {accuracy:.3f}")
print(f"F1 (macro): {macro_f1:.3f}")
print(f"Sens. avg : {macro_sen:.3f}")
print(f"Spec. avg : {macro_spec:.3f}")


In [ ]:
#  ▸ ROC eğrisi + temel metrikler (tek hücre)
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, auc,
                             confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize

# --- yollar & veri seti ---
MODEL_PATH = 'artifacts/best_vgg19_stage2.keras'
best_model = tf.keras.models.load_model(MODEL_PATH)

# test_ds ve class_names bellektesin diyelim; yoksa yeniden yükle:
# test_ds      = ...
# class_names  = ...

# ---------- 1) Tahminler ----------
y_true  = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
y_score = best_model.predict(test_ds, verbose=0)
y_pred  = np.argmax(y_score, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

# ---------- 2) ROC ----------
y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

fpr, tpr, roc_auc = {}, {}, {}
for i, c in enumerate(class_names):
    fpr[c], tpr[c], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[c]        = auc(fpr[c], tpr[c])

# mikro & makro
fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_score.ravel())
roc_auc["micro"]              = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[c] for c in class_names]))
mean_tpr = np.zeros_like(all_fpr)
for c in class_names:
    mean_tpr += np.interp(all_fpr, fpr[c], tpr[c])
mean_tpr /= len(class_names)
fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
roc_auc["macro"]           = auc(fpr["macro"], tpr["macro"])

# ---------- 3) Çizim ----------
plt.figure(figsize=(6,5))
plt.plot(fpr["micro"], tpr["micro"], ':',  lw=2,
         label=f"micro-avg AUC = {roc_auc['micro']:.2f}")
plt.plot(fpr["macro"], tpr["macro"], '-.', lw=2,
         label=f"macro-avg AUC = {roc_auc['macro']:.2f}")
for c in class_names:
    plt.plot(fpr[c], tpr[c], label=f"{c} (AUC = {roc_auc[c]:.2f})")

plt.plot([0,1],[0,1],'k--',lw=0.8)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Eğrileri • Test Kümesi"); plt.grid(linestyle='--', alpha=0.5)
plt.legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()


In [ ]:
import random, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg19 import preprocess_input

# --- 1) Modeli ve test generator'ı hazır sayıyoruz ---
best_model = tf.keras.models.load_model(
    'artifacts/best_vgg19_stage2.keras'
)

target_names = [cls for cls, _ in
                sorted(test_generator.class_indices.items(), key=lambda x: x[1])]

# --- 2) Rastgele bir test görseli seç ---
file_paths = test_generator.filepaths
rand_idx   = random.randint(0, len(file_paths) - 1)
img_path   = file_paths[rand_idx]

# --- 3) Görseli oku ve **aynı preprocessing** uygula ---
IMG_SIZE = 224     # eğitimde kullandığın boyut
img      = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
x        = image.img_to_array(img)          # 0-255 aralığında
x        = np.expand_dims(x, axis=0)
x        = preprocess_input(x)              # VGG19 ile aynı ölçekleme

# --- 4) Tahmin ---
pred_probs     = best_model.predict(x, verbose=0)[0]
predicted_idx  = np.argmax(pred_probs)
predicted_cls  = target_names[predicted_idx]

true_idx  = test_generator.classes[rand_idx]
true_cls  = target_names[true_idx]

# --- 5) Sonuç ---
print(f"Gerçek:   {true_cls}")
print(f"Tahmin:   {predicted_cls}\n")
for i, name in enumerate(target_names):
    print(f"{name}: {pred_probs[i]:.2f}")

plt.imshow(img); plt.axis('off'); plt.show()


In [ ]:
import shutil, os

# ------- 1) HDF5 (.h5) olarak kaydet -------
save_path_h5 = 'artifacts/vgg19_son.h5'   # ← dosya adını dilediğin gibi değiştir
model.save(save_path_h5, save_format='h5')
print(f"✅ Model (HDF5) kaydedildi → {save_path_h5}")

# ------- 2) (İsteğe bağlı) Çalışma dizinine kopyala -------
dest_path = 'artifacts/vgg19_son.h5'
shutil.copy(save_path_h5, dest_path)
print(f"📂 Çalışma dizinine de kopyalandı → {dest_path}")

# Kontrol: dosya boyutu göster
print("Dosya boyutu:", round(os.path.getsize(save_path_h5)/1e6, 2), "MB")



In [ ]:
# ============ YOL AYARLARI ============
SAVE_DIR  = 'artifacts'         # .keras / .h5 dosyalarının gideceği klasör
MODEL_BASENAME = 'vgg19_son'                       # dosya adı kökü (uzantılar otomatik eklenir)
# ======================================


import os, tensorflow as tf
os.makedirs(SAVE_DIR, exist_ok=True)

# ----- 1) .keras (yeni format) -----
keras_path = os.path.join(SAVE_DIR, MODEL_BASENAME + '.keras')
model.save(keras_path)           # compile=True/False fark etmez
print('✅ Kaydedildi:', keras_path)

# ----- 2) .h5 (legacy) -----
h5_path = os.path.join(SAVE_DIR, MODEL_BASENAME + '.h5')
model.save(h5_path, save_format='h5')
print('✅ Kaydedildi:', h5_path)


In [ ]:
# hist1 + hist2 → tek sözlük
full_hist = {k: hist1.history[k] + hist2.history[k] for k in hist1.history}

import matplotlib.pyplot as plt
plt.figure(figsize=(10,4))

# -------- Accuracy --------
plt.subplot(1,2,1)
plt.plot(full_hist['accuracy'],     label='Train acc')
plt.plot(full_hist['val_accuracy'], label='Val acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy')
plt.legend(); plt.grid(True)

# -------- Loss --------
plt.subplot(1,2,2)
plt.plot(full_hist['loss'],     label='Train loss')
plt.plot(full_hist['val_loss'], label='Val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# (Hazırlık) test_ds henüz tanımlı değilse AÇ/KULLAN
# ---------------------------------------------------------
# DATA_DIR = 'data/split'
# IMG_SIZE, BATCH_SIZE, SEED = 224, 32, 42
#
# from tensorflow.keras.applications.vgg19 import preprocess_input
# test_raw = tf.keras.utils.image_dataset_from_directory(
#     os.path.join(DATA_DIR, 'test'),
#     shuffle=False, seed=SEED,
#     image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
#     label_mode='categorical')
# class_names = test_raw.class_names
# NUM_CLASSES = len(class_names)
# test_ds = (test_raw
#            .map(lambda x, y: (preprocess_input(x), y))
#            .cache()
#            .prefetch(tf.data.AUTOTUNE))
# =========================================================

# 1) En iyi ağırlıkları yükle
best_model = tf.keras.models.load_model(
    'artifacts/best_vgg19_stage2.keras',
    compile=False
)
best_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])
print("✅ best_vgg19_stage2.keras yüklendi ve derlendi.")

# 2) Test kümesinde değerlendirme
test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc:.4f}")

# 3) Tahminler
y_pred_probs = best_model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])

# 4) Sınıf raporu
from sklearn.metrics import classification_report, confusion_matrix
print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names))

# 5) Confusion matrix görselleştir
import matplotlib.pyplot as plt, seaborn as sns
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Tahmin'); plt.ylabel('Gerçek'); plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()


In [ ]:
import os, time, tensorflow as tf, shutil

ckpt_path = 'artifacts/best_vgg19_stage2.keras'

# 1) ESKİ dosyayı sil
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print("🗑️  Eski dosya silindi.")
else:
    print("ℹ️  Eski dosya zaten yok.")

# 2) YENİ ağırlıkları kaydet
#    - bellekte 'best_model' değişkeni en iyi ağırlıkları içermeli
best_model.save(ckpt_path)          # overwrite artık mümkün
print("✅ Yeni ağırlıklar kaydedildi:", ckpt_path)

# 3) Drive’ın yazmayı bitirmesini bekleyip zaman damgası göster
time.sleep(2)                       # kısa bekleme
ts = time.ctime(os.path.getmtime(ckpt_path))
size = round(os.path.getsize(ckpt_path)/1e6, 2)
print(f"🕒 Güncel zaman damgası: {ts} | Boyut: {size} MB")

# 4) Hızlı doğrulama – yükle & test accuracy
model_check = tf.keras.models.load_model(ckpt_path, compile=False)
model_check.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
_, acc = model_check.evaluate(test_ds, verbose=0)
print(f"🔍 Kontrol Test Accuracy: {acc:.4f}")


In [ ]:
# ---------------- YOLUMU BELİRLE ----------------
DRIVE_PATH = 'artifacts/vgg19_best_full.keras'
# ------------------------------------------------


# 1) Bellekteki en iyi modelin adı best_model ise ↓
TMP_PATH = 'artifacts/vgg19_best_full.keras'   # geçici yerel dosya
best_model.save(TMP_PATH, include_optimizer=True)   # ≈170-180 MB

# 2) Drive’a kopyala
import shutil, os, time
shutil.copy(TMP_PATH, DRIVE_PATH)
time.sleep(2)                                   # senkron bekle

size_mb = round(os.path.getsize(DRIVE_PATH)/1e6, 2)
print(f"✅ KAYIT TAMAM   → {DRIVE_PATH}\n   Dosya boyutu: {size_mb} MB")
